# Phase 0: Environment Setup, Configuration Loading & Decision Tracking

In this notebook, we walk through Phase 0 step-by-step with small, focused cells.
Every cell does one clear action so you can explain each line in an interview.

In [ ]:
# 1. Core library imports
import os
import tempfile
from pathlib import Path
from typing import Any, Dict
import yaml

In [ ]:
# 2. Check current working directory
current_dir = Path().resolve()
print(f"Current working directory: {current_dir}")

In [ ]:
# 3. Anchor project root safely (avoids FileNotFoundError in tests/Docker)
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
print(f"Project root anchored at:  {project_root}")

In [ ]:
# 4. Define paths to configs and data directories
configs_dir = project_root / "configs"
data_dir = project_root / "data"
print(f"Configs directory: {configs_dir}")
print(f"Data directory:    {data_dir}")

In [ ]:
# 5. Verify that required project directories exist
assert configs_dir.exists(), "configs/ directory must exist!"
assert data_dir.exists(), "data/ directory must exist!"
print("Directory check passed successfully!")

### Step 2: Model Pricing Configuration (`configs/prices.yaml`)
Why externalize pricing? Storing dated token costs in YAML ensures 100% reproducible cost-per-1k-queries calculations.

In [ ]:
# 6. Define path to prices.yaml
prices_path = configs_dir / "prices.yaml"
print(f"Prices file: {prices_path}")

In [ ]:
# 7. Read raw YAML content
with open(prices_path, "r", encoding="utf-8") as f:
    prices_data = yaml.safe_load(f)

In [ ]:
# 8. Inspect pricing metadata
print(f"Pricing effective date: {prices_data.get('effective_date')}")
print(f"Currency:               {prices_data.get('currency')}")

In [ ]:
# 9. Inspect configured models
models_list = list(prices_data["models"].keys())
print(f"Configured models ({len(models_list)}): {models_list}")

In [ ]:
# 10. Display input and output costs per 1M tokens for each model
for model_name, info in prices_data["models"].items():
    inp = info["input_cost_per_1m"]
    out = info["output_cost_per_1m"]
    print(f"{model_name:20s} | Input: ${inp:6.3f} / 1M | Output: ${out:6.3f} / 1M")

### Step 3: Safe YAML Config Loader (`src/config_loader.py`)
We build the `load_yaml` function line-by-line with defensive guards.

In [ ]:
# 11. Define the load_yaml function with defensive checks
def load_yaml(file_path: Path | str) -> Dict[str, Any]:
    # Convert string to Path
    path = Path(file_path)
    
    # Anchor relative path to project root
    if not path.is_absolute():
        path = project_root / path
        
    # Guard 1: Verify file exists
    if not path.exists():
        raise FileNotFoundError(f"Configuration file not found at: {path}")
        
    # Read and parse YAML
    with open(path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)
        
    # Guard 2: Reject empty YAML file
    if data is None:
        raise ValueError(f"Configuration file is empty: {path}")
        
    # Guard 3: Reject non-dictionary root (must be key-value mapping)
    if not isinstance(data, dict):
        raise ValueError(f"Configuration file root must be a mapping/dict: {path}")
        
    return data

In [ ]:
# 12. Test load_yaml on our prices config
loaded_config = load_yaml("configs/prices.yaml")
print(f"Successfully loaded config with {len(loaded_config['models'])} models!")

### Step 4: Testing Error Handling & Edge Cases
In interviews, robust error handling separates junior scripts from production code.

In [ ]:
# 13. Test Guard 1: Non-existent file
try:
    load_yaml("configs/non_existent.yaml")
except FileNotFoundError as e:
    print(f"Passed Guard 1: Caught expected FileNotFoundError -> {e}")

In [ ]:
# 14. Test Guard 2: Empty YAML file
with tempfile.NamedTemporaryFile(suffix=".yaml", mode="w", delete=False) as tmp:
    tmp_empty = Path(tmp.name)

try:
    load_yaml(tmp_empty)
except ValueError as e:
    print(f"Passed Guard 2: Caught expected ValueError on empty file -> {e}")
finally:
    tmp_empty.unlink()

In [ ]:
# 15. Test Guard 3: List-based YAML file (invalid mapping root)
with tempfile.NamedTemporaryFile(suffix=".yaml", mode="w", delete=False) as tmp:
    tmp.write("- item1\n- item2\n")
    tmp_list = Path(tmp.name)

try:
    load_yaml(tmp_list)
except ValueError as e:
    print(f"Passed Guard 3: Caught expected ValueError on list root -> {e}")
finally:
    tmp_list.unlink()

### Step 5: Pre-Registered Success Target & Interview Summary

- **Target:** Field-level F1 within 3 points of B3 (large LLM), lower bound of 95% bootstrap CI > -3.0, at < 25% of B3 cost.
- **Dev Set (~60):** For tuning and calibration only.
- **Test Set (~140):** Locked and scored once per milestone.